# Final held-out comparison: original Qwen vs frozen epoch-2 adapter

Run this only after model selection is finished. We have selected **checkpoint-32 (epoch 2)** using validation loss. This notebook runs both original Qwen and that adapter on the same **66 test tickets**. It does not train, change settings, or publish anything.

Model: `Qwen/Qwen3-0.6B`, revision `c1899de289a04d12100db370d81485cdf75e47ca`. Prompt, test data, adapter files, and scoring code are fingerprinted in `training/final_test_reference.json` before evaluation.

Keep your original notebook/runtime connected if possible. This notebook also supports a new runtime using the local adapter backup. No Drive access is needed. If local files changed, rebuild packages with `python3 -m scripts.prepare_final_test` before uploading.

**Important:** all results are from a small AI-authored dataset. Do not tune using these final test results and continue describing the same test set as independent evidence.


## 1. Connect a T4 GPU

Select a GPU runtime and check that PyTorch can see it. If continuing in the existing training notebook, you can append these cells there in order.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Upload final-test code

Select **colab-final-test-project.zip**. This contains code/data and the frozen experiment reference, without weights or secrets. It can update the existing project code without deleting your training artifacts. It does not contain your adapter; that is handled separately in Step 4.

In [ ]:
from pathlib import Path
from google.colab import files
from zipfile import ZipFile
import io
import shutil
import os

uploaded = files.upload()
assert len(uploaded) == 1, "Upload colab-final-test-project.zip only."
with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        assert not relative.is_absolute() and ".." not in relative.parts
        assert len(relative.parts) >= 2 and relative.parts[0] == "ticket-triage"
        inside = Path(*relative.parts[1:])
        assert inside.parts[0] in ("evaluation", "training", "data", "README.md")
        assert (member.external_attr >> 16) & 0o170000 != 0o120000
        if member.is_dir():
            continue
        target = Path("/content/ticket-triage") / inside
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
ROOT = Path("/content/ticket-triage")
os.chdir(ROOT)
print("Final-test code ready:", ROOT)
del uploaded


## 3. Use the same library versions

Install the same pinned Transformers/PEFT/Accelerate versions used for training. No new training occurs. If Colab requests a restart, restart then rerun this cell and the following cells.

In [ ]:
from pathlib import Path
import os
ROOT = Path("/content/ticket-triage")
os.chdir(ROOT)
%pip install -q -r training/requirements-training.txt
import sys
import subprocess

def run(*arguments):
    process = subprocess.Popen([sys.executable, "-u", *map(str, arguments)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Command failed; read the error above and stop here.")


## 4. Locate the frozen adapter and check the experiment

If your training folder is still present, this cell uses it. Otherwise, select **final-test-adapter.zip** from the local `artifacts/` folder when asked. That restore package contains only the selected adapter and tokenizer/configuration files, not original Qwen weights.

The check verifies file hashes without running predictions. It rejects a changed adapter, prompt, test file, or scoring implementation. A fresh output folder preserves this final evaluation separately from validation results.


In [ ]:
from google.colab import files
from zipfile import ZipFile
from datetime import datetime, timezone
import io
import shutil

BUNDLE = ROOT / "artifacts/20261115T073319089436Z"
if not (BUNDLE / "training/adapter/adapter_model.safetensors").is_file():
    BUNDLE = Path("/content/final-adapter-bundle")
    if not (BUNDLE / "training/adapter/adapter_model.safetensors").is_file():
        print("Upload final-test-adapter.zip now.")
        uploaded = files.upload()
        assert len(uploaded) == 1, "Upload only the adapter restore ZIP."
        with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
            for member in archive.infolist():
                relative = Path(member.filename)
                assert not relative.is_absolute() and ".." not in relative.parts
                assert relative.parts[:2] == ("training", "adapter")
                assert (member.external_attr >> 16) & 0o170000 != 0o120000
                if member.is_dir():
                    continue
                target = BUNDLE / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
        del uploaded
FINAL_DIR = ROOT / "artifacts" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-final-test")
run("-m", "evaluation.final_test", "--bundle", BUNDLE, "--output-dir", FINAL_DIR, "--check-only")
print("Adapter verified. Final output folder:", FINAL_DIR)


## 5. Run both models on all 66 test tickets

This is the final held-out evaluation. The script loads original Qwen, generates answers, scores them, and then repeats with the frozen selected adapter. Both use the same prompt, model commit, device, dtype, libraries, and generation settings. The inference messages contain only instructions and ticket text; gold answers are used afterward to calculate scores.

The script prints the final comparison and saves raw predictions, metrics, a manifest, and a human-review CSV. Invalid-schema answers count as incorrect across task metrics. Do not change the adapter or settings based on these results and then claim the same test set is untouched.


In [ ]:
run("-m", "evaluation.final_test", "--bundle", BUNDLE, "--output-dir", FINAL_DIR)


## 5b. View and export the portfolio dashboard

Run this after Step 5 completes. It reads the actual saved final-test results and displays overall scores, per-category F1, and original/adapted confusion matrices. No example or fabricated scores are used.

The PNG is suitable for a portfolio screenshot; a PDF is also saved. Keep the dataset limitation notes visible when sharing. This chart does not measure impact-summary factuality.


In [ ]:
# Matplotlib is normally available in Colab. Install only if missing.
import importlib.util
if importlib.util.find_spec("matplotlib") is None:
    run("-m", "pip", "install", "matplotlib")
run("-m", "evaluation.dashboard", "--results-dir", FINAL_DIR)
from IPython.display import Image, display
display(Image(filename=str(FINAL_DIR / "final_test_dashboard.png")))

# Refresh the results ZIP to include the dashboard exports.
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile(FINAL_DIR.parent / (FINAL_DIR.name + ".zip"), "w", ZIP_DEFLATED) as archive:
    for path in FINAL_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(FINAL_DIR))


## 6. Download the final results immediately

Download before disconnecting: Colab storage is temporary. Save and extract this ZIP under your local `artifacts/` folder. It contains results, not adapter or base weights; keep your earlier adapter ZIP too.

- `base-test/predictions.jsonl` and `adapted-test/predictions.jsonl`: raw answers and timings.
- Each model's `metrics.json`: overall and per-category scores/confusion matrix.
- `comparison.json`: original vs adapted scores and differences.
- `final_test_manifest.json`: frozen model/adapter/data/settings identity and run status.
- `human_review.csv`: tickets, expected answers, both predictions, and blank review fields.

Review impact summaries for factuality and completeness before making extraction-quality claims. Team is derived from category. The automated scores do not measure summary meaning, and no human review ratings are fabricated.


In [ ]:
from google.colab import files
results_zip = FINAL_DIR.parent / (FINAL_DIR.name + ".zip")
assert results_zip.is_file(), "Complete Step 5 first."
files.download(str(results_zip))
print("Keep this final results ZIP AND your earlier adapter backup.")
print("Results path:", results_zip)
